# MMUbiPred–Context Residual Hybrid: external assessment

This CPU-capable notebook evaluates the previously frozen residual hybrid on the existing 286-site and 740-site external cohorts. It reuses checksum-locked exact-MMUbiPred and context probabilities, performs only frozen local-expert inference, and applies the unchanged residual stacker at threshold 0.5. No model is trained or tuned. Because both cohorts have already been inspected, all results are post-primary exploratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.4,<2', 'scipy>=1.11,<2'], check=True)

In [ ]:
import json, torch
print('Device available:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('A GPU is optional; only the small local expert is inferred.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Locate frozen artifacts

The two existing external evaluation directories and the residual-hybrid refit are required in Drive. The paper H5 file, TensorFlow, ESM-2 checkpoint, and sequence caches are unnecessary because their saved probabilities are reused.

In [ ]:
ANALYSIS_CONFIG = Path('experiment/configs/external_residual_hybrid_exploratory_v1.json')
DEVELOPMENT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
HYBRID_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-full-refit-exploratory-v1')

COHORTS = {
    'dbptm_ptmgpt2_286': {
        'cohort_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/cohort'),
        'source_evaluation_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/evaluation'),
        'output_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/residual-hybrid-exploratory-v1'),
    },
    'disjoint_dbptm_740': {
        'cohort_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v2/cohort'),
        'source_evaluation_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v2/evaluation'),
        'output_dir': Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v2/residual-hybrid-exploratory-v1'),
    },
}

required = [
    DEVELOPMENT_RUN_DIR / 'stabilized_summary.json',
    HYBRID_RUN_DIR / 'best.pt',
    HYBRID_RUN_DIR / 'refit_manifest.json',
    HYBRID_RUN_DIR / 'refit_summary.json',
]
for paths in COHORTS.values():
    required.extend([
        paths['cohort_dir'] / 'external_cohort.tsv',
        paths['cohort_dir'] / 'cohort_lock.json',
        paths['source_evaluation_dir'] / 'external_test_metrics.json',
        paths['source_evaluation_dir'] / 'external_test_predictions.npz',
        paths['source_evaluation_dir'] / 'external_test_manifest.json',
    ])
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required Drive artifacts:\n' + '\n'.join(missing))
print('All frozen residual-hybrid and external artifacts were found.')

## Evaluate both external cohorts

Completed outputs are not repeated. Local inference is brief relative to the 10,000-replicate protein-cluster bootstrap, which accounts for most runtime.

In [ ]:
for cohort_key, paths in COHORTS.items():
    paths['output_dir'].mkdir(parents=True, exist_ok=True)
    result_path = paths['output_dir'] / 'residual_hybrid_external_metrics.json'
    if result_path.exists():
        print('Existing completed result; skipped:', cohort_key)
        continue
    run_live([
        sys.executable, '-u',
        'experiment/scripts/evaluate_external_residual_hybrid.py',
        '--analysis-config', ANALYSIS_CONFIG,
        '--cohort-key', cohort_key,
        '--cohort-dir', paths['cohort_dir'],
        '--source-evaluation-dir', paths['source_evaluation_dir'],
        '--development-run-dir', DEVELOPMENT_RUN_DIR,
        '--hybrid-run-dir', HYBRID_RUN_DIR,
        '--output-dir', paths['output_dir'],
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-post-primary-exploration',
    ])

## Display the comparisons

The two displayed summaries form the external-assessment record. Confidence intervals, rather than point estimates alone, determine whether an apparent difference is stable.

In [ ]:
for cohort_key, paths in COHORTS.items():
    result = json.loads((paths['output_dir'] / 'residual_hybrid_external_metrics.json').read_text())
    print('\n===', cohort_key, '===')
    display({
        'analysis_role': result['analysis_role'],
        'comparison_valid': result['comparison_valid'],
        'confirmatory_claim_allowed': result['confirmatory_claim_allowed'],
        'cohort': result['cohort'],
        'architecture': result['architecture'],
    })
    display(result['exact_released_mmubipred'])
    display(result['residual_hybrid'])
    display(result['residual_hybrid_minus_exact_mmubipred'])
    display(result['paired_residual_hybrid_vs_exact_mmubipred']['bootstrap']['metrics'])